# Unit 3｜Pandas 資料處理


## 目錄
1. [安裝與匯入](#3-0)
2. [DataFrame 基礎](#3-1)
3. [資料選取](#3-2)
4. [資料清理](#3-3)
5. [資料轉換](#3-4)

---
<a id='3-0'></a>
### 3-0 安裝與匯入

In [2]:
import pandas as pd
import numpy as np
print(f'pandas 版本:{pd.__version__}')

pandas 版本:2.2.2


---
## 第一節：DataFrame 基礎、資料選取、資料清理

<a id='3-1'></a>
### 3-1 DataFrame 基礎

pandas 的核心物件：
- **Series**：一維，像是有標籤的 list
- **DataFrame**：二維，像是有標籤的表格（最常用）

In [ ]:
# 從dict建立DataFrame

data = {'name':['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':[20, 22, 21, 23, 20],
    'score':[85, 75, 92, 55, 88],
    'grade':['B', 'C', 'A', 'F', 'B'],
    'passed':[True, True, True, False, True]}

type(data)
df = pd.DataFrame(data)
df

,name,age,score,grade,passed
0,Alice,20,85,B,True
1,Bob,22,75,C,True
2,Carol,21,92,A,True
3,Dave,23,55,F,False
4,Eve,20,88,B,True


In [ ]:
# 基本資訊查詢

df.shape # 用來查詢DataFrame的形狀(列數和欄數)
df.dtypes # 用來查詢資料的型別
df.columns # 用來查詢DataFrame的欄位名稱
df.index # 用來查詢DataFrame的index

# 快速預覽
df.head(2) # 顯示前五筆資料
df.tail(2) # 顯示後五筆資料
df.describe() # 計算敘述性統計表

,age,score
count,5.00000,5.000000
mean,21.20000,79.000000
std,1.30384,14.815532
min,20.00000,55.000000
25%,20.00000,75.000000
50%,21.00000,85.000000
75%,22.00000,88.000000
max,23.00000,92.000000


In [ ]:
# 讀取 CSV/Excel

# 將資料儲存成CSV檔
df.to_csv('students.csv', index=False, encoding='utf8')

# 將CSV讀取至變數內
df1 = pd.read_csv('students.csv', encoding='utf8')
df1
df2 = df1['score']
df2.mean() # 算平均數
df2.max() # 找最大值
df2.min() # 找最小值
df2.round(2) # 取小數

,score
0,85
1,75
2,92
3,55
4,88


<a id='3-2'></a>
### 3-2 資料選取

In [ ]:
# 讀取檔案
stock = pd.read_csv('2327.csv', encoding='big5')
stock.head(20)
stock.tail(20)

# 欄位的更名
stock = stock.rename(columns={'證券代碼':'coid', '簡稱':'name', '年月日':'date',
            '開盤價(元)':'open', '最高價(元)':'high', '最低價(元)':'low',
            '收盤價(元)':'close', '成交量(千股)':'volume'})
stock

# 欄位的刪除
stock = stock.drop(columns=['coid', 'name'])
stock

# 時間格式的變更
stock['date'] = pd.to_datetime(stock['date'], format='%Y%m%d')
stock

# 將日期設為索引
stock = stock.set_index('date')
stock.head()

stock.to_csv('2327_Final.csv', encoding='utf8')

In [ ]:
# 一般寫法:選取欄位
stock['close'] # 選取單欄
stock[['close', 'high']] # 選取多欄

# 用loc方法，用列和欄的名稱進行選取
stock.loc['2020-01-01':'2022-01-02', 'close'] # 選取一段時間的收盤價
stock.loc[:, 'close'] # 跟上述的選取單欄的結果是相同的
stock.loc['2021-01-04', 'close'] # 選取某一日的收盤價
stock.loc['2021-01-04', ['close', 'high']] # 選取某一日的收盤價和最高價
stock.loc['2020-01-01':'2022-01-02', ['close', 'high']]# 選取一段時間的收盤價和最高價
stock.loc[:, ['close','high']] # 跟上述的選取多欄的結果是相同的

# 用iloc方法進行欄位選取，透過整數位置的方式進行選取
stock.iloc[:, 3] # 使用整數指定close欄位並抓取
stock.iloc[0:10, 3] # 使用整數指定index並抓取
stock.iloc[:, [1, 2, 3]] # 使用整數進行多欄選取
stock.iloc[1, [3]] # 使用整數進行單列單欄選取
stock.iloc[1, [3, 2]] # 使用整數進行單列多欄選取

# 條件篩選(布林遮罩)
# 單一條件的篩選
stock[stock['close'] > 100]

# 多個條件的篩選
stock[(stock['close'] > 100) & (stock['open'] >= 95)]

#### 🖊️ 隨堂練習 3-2
從 `df` 中篩選出：
1. 分數介於 70～90 之間（含）的學生（用 `between`）
2. 年齡為 20 或 22 歲的學生
3. 只顯示 name 和 score 兩欄

In [ ]:
# 找出分數介於70~90之間的同學
make_score = df['score'].between(70, 90)
df[make_score][['name','score']]

# 年齡20或22
mask_age = df['age'].isin([20, 22])
df[mask_age][['name', 'age']]

# 合併條件:分數70-90並且年齡20或22，只顯示name和score
df[make_score & mask_age][['name', 'score']]

<a id='3-3'></a>
### 3-3 資料清理

In [ ]:
# 建立一個含有問題的資料及

dirty = pd.DataFrame({'name':['Alice', 'Bob', None, 'Dave', 'Bob', 'Eve'],
            'age':[20, 22, 21, None, 22, 20],
            'score':[85.0, 78.0,92.0 ,55.0, 78.0, None],
            'dept':[' Info', 'Fin', 'Info', 'Fin', 'Fin', 'Info']})
dirty

,name,age,score,dept
0,Alice,20.0,85.0,Info
1,Bob,22.0,78.0,Fin
2,None,21.0,92.0,Info
3,Dave,NaN,55.0,Fin
4,Bob,22.0,78.0,Fin
5,Eve,20.0,NaN,Info


In [ ]:
# 檢查遺漏值

dirty.isnull().sum() # 檢查各欄空值的數量
dirty.isnull().mean().round(3) # 檢查空值的比例

,0
name,1
age,1
score,1
dept,0


In [ ]:
# 處理空值
df_clean = dirty.copy() # copy()用來完整複製變數內容
df_clean

print(dirty)
df_clean.dropna() # 任何一個欄位有空值就刪除整列
df_clean['name'] = df_clean['name'].fillna('Alice') # 填補空值:用固定值的方法填補
df_clean
# df_clean['age'] = df_clean['age'].fillna(20)
# df_clean

df_clean['score'] = df_clean['score'].fillna(df_clean['score'].mean()) # 用該欄的平均值進行填補
df_clean

df_clean['age'] = df_clean['age'].ffill() # 用前一筆資料進行空值填補
df_clean

df_clean.isnull().sum()

    name   age  score   dept
0  Alice  20.0   85.0   Info
1    Bob  22.0   78.0    Fin
2   None  21.0   92.0   Info
3   Dave   NaN   55.0    Fin
4    Bob  22.0   78.0    Fin
5    Eve  20.0    NaN   Info


,0
name,0
age,0
score,0
dept,0


In [ ]:
dirty
dirty.duplicated().sum() # 顯示重複的列數

# 刪除重複列
dirty.drop_duplicates()

# 字串欄位的清理
dirty['deptartment'].str.strip() # 去除字串間的前後空白

# 欄位重新命名
dirty = dirty.rename(columns={'dept':'deptartment'})
dirty

# 欄位排序
dirty = dirty.sort_values('score', ascending=False)
dirty

# 重設索引
dirty = dirty.reset_index(drop=True)
dirty

,name,age,score,dept
0,Alice,20.0,85.0,Info
1,Bob,22.0,78.0,Fin
2,None,21.0,92.0,Info
3,Dave,NaN,55.0,Fin
5,Eve,20.0,NaN,Info


#### 🖊️ 隨堂練習 3-3
對以下資料進行完整清理流程，輸出一份乾淨的 DataFrame：
1. 填補 `salary` 缺漏值（用平均值）
2. 刪除 `name` 為空的列
3. 去除重複列
4. 清理 `dept` 欄的前後空白
5. 重設 index

In [ ]:
messy = pd.DataFrame({
    "name":   ["Alice", None, "Carol", "Dave", "Alice"],
    "dept":   [" IT ", " HR", "IT", " HR ", " IT "],
    "salary": [60000, 55000, None, 58000, 60000]
})

# 1.
messy['salary'] = messy['salary'].fillna(messy['salary'].mean())

# 2.
messy = messy.dropna(subset=['name'])

# 3.
messy = messy.drop_duplicates()

# 4.
messy['dept'] = messy['dept'].str.strip()
messy

# 5.
messy = messy.reset_index(drop=True)
messy


,name,dept,salary
0,Alice,IT,60000.0
1,Carol,IT,58250.0
2,Dave,HR,58000.0


---
## 第二節：資料轉換、合併、輸出

<a id='3-4'></a>
### 3-4 資料轉換

In [40]:
# 準備資料集

df = pd.DataFrame({'name':['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
          'dept':['IT', 'HR', 'IT', 'HR', 'IT', 'Finance'],
          'salary':[65000, 55000, 70000, 58000, 62000, 80000],
          'years':[3, 5, 7, 2, 4, 10],
          'score':[88, 75, 92, 68, 85, 95]})

df

,name,dept,salary,years,score
0,Alice,IT,65000,3,88
1,Bob,HR,55000,5,75
2,Carol,IT,70000,7,92
3,Dave,HR,58000,2,68
4,Eve,IT,62000,4,85
5,Frank,Finance,80000,10,95


In [42]:
df['monthly'] = round(df['salary'] / 12, 2)
df
df['bonus'] = df['salary'] * 0.1

# step1:首先判斷規則(years >= 5 就是資深，否則就是一般)
# step2:用apply將判斷規則套用到years欄位的每一列
# step3:將結果存入新欄位level
# lambda是Python匿名函數，用寫一個沒有名字的函數，它通常都是用在函數邏輯很簡單的狀況下面
df['level'] = df['years'].apply(lambda y: '資深' if y >= 5 else '一般')
df
# 用一般自訂函數的寫法來寫
# 先定義判斷函數
def get_level(y):
  if y >= 5:
    return '資深'
  else:
    return '一般'

# 再用apply套用到years欄位的每一列
df['level'] = df['years'].apply(get_level)
df

# apply函數:主要是用來對欄位套用自訂函數

def score_to_grade(s):
  if s >= 90:
    return 'A'
  elif s >= 80:
    return 'B'
  elif s >= 70:
    return 'C'
  else:
    return 'F'
df['grade'] = df['score'].apply(score_to_grade)
df

,name,dept,salary,years,score,monthly,bonus,level,grade
0,Alice,IT,65000,3,88,5416.67,6500.0,一般,B
1,Bob,HR,55000,5,75,4583.33,5500.0,資深,C
2,Carol,IT,70000,7,92,5833.33,7000.0,資深,A
3,Dave,HR,58000,2,68,4833.33,5800.0,一般,F
4,Eve,IT,62000,4,85,5166.67,6200.0,一般,B
5,Frank,Finance,80000,10,95,6666.67,8000.0,資深,A


In [57]:
# groupby:分組統計，運作邏輯是先Split->Apply->Combin，先拆分，再套用，再合併
# df.groupby('欄位名稱')['要計算的欄位'].統計方法()

print('各部門的平均薪資')
df.groupby('dept')['salary'].mean().round(0)

print('各部門多項統計')
df.groupby('dept')[['salary', 'score']].agg(['mean', 'max', 'count']).round(0)

# groupby + 自訂agg
# 將df依照dept欄位進行分組並對每組計算多個統計指標
# agg函數代表aggregate(聚合)，通常都是使用在grouptby的後面，
# 目的是為了對每一組資料同時計算多個統計指標
result = df.groupby('dept').agg(
    人數 = ('name', 'count'),  # 計算每個部門有幾筆name
    平均薪資 = ('salary', 'mean'), # 計算每個部門salary的平均值
    最高薪資 = ('salary', 'max'),  # 計算每個部門salary的最大值
    平均考核 = ('score', 'mean') # 計算每個部門score的平均值
).round(0)
result

dept     years
Finance  10       80000.0
HR       2        58000.0
         5        55000.0
IT       3        65000.0
         4        62000.0
         7        70000.0
Name: salary, dtype: float64

In [34]:
# 排序

print('依薪資進行降序')
df.sort_values('salary', ascending=False)[['name', 'dept', 'salary']]

print('先依部門，再依薪資進行降序')
df.sort_values(['dept', 'salary'], ascending=[True, False])[['name', 'dept', 'salary']]

依薪資進行降序
先依部門，再依薪資進行降序


,name,dept,salary
5,Frank,Finance,80000
3,Dave,HR,58000
1,Bob,HR,55000
2,Carol,IT,70000
0,Alice,IT,65000
4,Eve,IT,62000


In [44]:
# 刪除欄位
df
# df = df.drop(columns=['monthly', 'bonus'])
df

,name,dept,salary,years,score,monthly,bonus,level,grade
0,Alice,IT,65000,3,88,5416.67,6500.0,一般,B
1,Bob,HR,55000,5,75,4583.33,5500.0,資深,C
2,Carol,IT,70000,7,92,5833.33,7000.0,資深,A
3,Dave,HR,58000,2,68,4833.33,5800.0,一般,F
4,Eve,IT,62000,4,85,5166.67,6200.0,一般,B
5,Frank,Finance,80000,10,95,6666.67,8000.0,資深,A


#### 🖊️ 隨堂練習 3-4
在 `df` 基礎上完成：
1. 新增欄位 `total_comp`（薪資 + 獎金），獎金規則：年資 >= 5 給 15%，否則給 8%
2. 用 groupby 統計各部門的 total_comp 平均值與人數
3. 將結果依平均值降序排列

In [50]:
# 1. 新增total_comp
df['bonus_rate'] = df['years'].apply(lambda y: 0.15 if y >= 5 else 0.08)
df['total_comp'] = df['salary']*(1+df['bonus_rate'])
df

# 2. 用groupby統計各部門的total_comp平均值和人數
result = df.groupby('dept').agg(
    人數 = ('name', 'count'),  # 計算每個部門有幾筆name
    平均總薪資 = ('total_comp', 'mean'), # 計算每個部門總薪酬的平均值
).round(0)
result

# 3. 依照總薪酬的平均值降序排列
result = result.sort_values('平均總薪資', ascending=False)
result

,人數,平均總薪資
dept,,
Finance,1,92000.0
IT,3,72553.0
HR,2,62945.0


<a id='3-5'></a>
### 3-5 合併資料

In [59]:
df_left = pd.DataFrame({'id':[1, 2, 3, 4, 5],
             'name':['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
              'dept':['IT', 'HR', 'IT', 'HR', 'IT']})
df_left

df_right = pd.DataFrame({'id':[1, 2, 3, 6],
              'salary':[65000, 55000, 70000, 80000],
              'score':[88, 75, 92, 95]})
df_right

,id,salary,score
0,1,65000,88
1,2,55000,75
2,3,70000,92
3,6,80000,95


In [64]:
# merage:用來把兩個DataFrame依照共同的欄位(Key值)合併成一張表，概念類似SQL的JOIN

print(df_left)
print(df_right)

# print('inner join(只保留兩邊都有的id)')
pd.merge(df_left, df_right, on='id', how='inner')

# print('left join(保留左表所有的列)')
pd.merge(df_left, df_right, on='id', how='left')

# print('right join(保留右表所有的列)')
pd.merge(df_left, df_right, on='id', how='right')

   id   name dept
0   1  Alice   IT
1   2    Bob   HR
2   3  Carol   IT
3   4   Dave   HR
4   5    Eve   IT
   id  salary  score
0   1   65000     88
1   2   55000     75
2   3   70000     92
3   6   80000     95


,id,name,dept,salary,score
0,1,Alice,IT,65000,88
1,2,Bob,HR,55000,75
2,3,Carol,IT,70000,92
3,6,NaN,NaN,80000,95


In [69]:
# concat:用來把多個DataFrame單純的拼接在一起，跟merage()的不同之處在於，
# merage()需要有共同的欄位(Key值)進行比對

df_q1 = pd.DataFrame({'name':['Alice', 'Bob'], 'sales':[100, 200],
            'quarter':['Q1', 'Q1']})
print(df_q1)

df_q2 = pd.DataFrame({'name':['Carol', 'Dave'], 'sales':[150, 180],
            'quarter':['Q2', 'Q2']})
print(df_q2)

pd.concat([df_q1, df_q2], ignore_index=True)

    name  sales quarter
0  Alice    100      Q1
1    Bob    200      Q1
    name  sales quarter
0  Carol    150      Q2
1   Dave    180      Q2


,name,sales,quarter
0,Alice,100,Q1
1,Bob,200,Q1
2,Carol,150,Q2
3,Dave,180,Q2


#### 🖊️ 隨堂練習 3-5
合併以下兩個表，完成：
1. 用 left join 合併（以 `student_id` 為 key）
2. 計算每位學生的總分（math + english + science）
3. 依總分降序排列，只顯示 name、class、total 三欄

In [79]:
students_info = pd.DataFrame({
    "student_id": [101, 102, 103, 104],
    "name":       ["Amy", "Ben", "Cate", "Dan"],
    "class":      ["A", "B", "A", "B"]
})

students_score = pd.DataFrame({
    "student_id": [101, 102, 103],
    "math":       [88, 72, 95],
    "english":    [76, 85, 90],
    "science":    [82, 78, 88]
})

# print(students_info)
# print(students_score)

# 1.
merged = pd.merge(students_info, students_score, on='student_id', how='left')

# 2.
merged['total'] = merged['math']+merged['english']+merged['science']
merged

# 3.
merged.sort_values('total', ascending=False)[['name', 'class', 'total']].reset_index(drop=True)

,name,class,total
0,Cate,A,273.0
1,Amy,A,246.0
2,Ben,B,235.0
3,Dan,B,NaN


---
<a id='project'></a>
## 單元實作：問卷資料分析

整合本單元完整流程：讀取 → 清理 → 轉換 → 分組統計 → 輸出報表

In [80]:
import pandas as pd
import numpy as np

# 模擬問卷原始資料（含髒資料）
np.random.seed(42)
n = 30

raw = pd.DataFrame({
    "id":     range(1, n + 1),
    "gender": np.random.choice(["男", "女", None], n, p=[0.45, 0.45, 0.1]),
    "age":    np.random.choice([18, 19, 20, 21, 22, None], n),
    "dept":   np.random.choice([" 資工系", "財務系 ", "資工系", "財務系", "企管系"], n),
    "q1":     np.random.choice([1, 2, 3, 4, 5, None], n),   # 李克特 1~5
    "q2":     np.random.choice([1, 2, 3, 4, 5, None], n),
    "q3":     np.random.choice([1, 2, 3, 4, 5], n),
    "q4":     np.random.choice([1, 2, 3, 4, 5], n),
})

print(f"原始資料：{raw.shape}")
print(raw.head(8))

原始資料：(30, 8)
   id gender   age  dept q1 q2  q3  q4
0   1      男    20   財務系  1  3   4   4
1   2   None    22   企管系  3  1   4   4
2   3      女    18  財務系   3  2   4   2
3   4      女    19  財務系   1  2   4   3
4   5      男    21   財務系  3  4   3   1
5   6      男    18  財務系   5  5   2   5
6   7      男    21  財務系   2  3   4   1
7   8      女  None   財務系  2  1   1   1


In [81]:
# 步驟 1：清理
df = raw.copy()

# 刪除 gender 缺漏（關鍵變數不填補）
df = df.dropna(subset=["gender"])

# 填補 age 用眾數
df["age"] = df["age"].fillna(df["age"].mode()[0])

# 填補 q1、q2 用中位數
df["q1"] = df["q1"].fillna(df["q1"].median())
df["q2"] = df["q2"].fillna(df["q2"].median())

# 清理字串欄位
df["dept"] = df["dept"].str.strip()

# 重設 index
df = df.reset_index(drop=True)

print(f"清理後：{df.shape}，缺漏值：{df.isnull().sum().sum()} 個")

清理後：(28, 8)，缺漏值：0 個


/tmp/ipykernel_1427/2364459215.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["age"] = df["age"].fillna(df["age"].mode()[0])
/tmp/ipykernel_1427/2364459215.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["q1"] = df["q1"].fillna(df["q1"].median())
/tmp/ipykernel_1427/2364459215.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tr

In [82]:
# 步驟 2：衍生變數
q_cols = ["q1", "q2", "q3", "q4"]
df["avg_score"] = df[q_cols].mean(axis=1).round(2)

# 態度分組
conditions = [
    df["avg_score"] >= 4,
    df["avg_score"] >= 3,
]
df["attitude"] = np.select(conditions, ["正面", "中立"], default="負面")
df[["id", "gender", "dept", "avg_score", "attitude"]].head(8)

,id,gender,dept,avg_score,attitude
0,1,男,財務系,3.00,中立
1,3,女,財務系,2.75,負面
2,4,女,財務系,2.50,負面
3,5,男,財務系,2.75,負面
4,6,男,財務系,4.25,正面
5,7,男,財務系,2.50,負面
6,8,女,財務系,1.25,負面
7,9,女,財務系,2.25,負面


In [ ]:
# 步驟 3：分析
print("=== 整體描述統計 ===")
print(df[q_cols + ["avg_score"]].describe().round(2))

print("\n=== 各系態度分布 ===")
print(pd.crosstab(df["dept"], df["attitude"], margins=True))

print("\n=== 各系平均分數 ===")
print(df.groupby("dept")["avg_score"].agg(["mean", "count"]).round(2))

print("\n=== 性別 × 各題平均 ===")
print(df.groupby("gender")[q_cols].mean().round(2))

In [ ]:
# 步驟 4：輸出報表
df.to_csv("survey_clean.csv", index=False, encoding="utf-8-sig")

summary = df.groupby("dept").agg(
    人數     = ("id",        "count"),
    Q1平均   = ("q1",        "mean"),
    Q2平均   = ("q2",        "mean"),
    Q3平均   = ("q3",        "mean"),
    Q4平均   = ("q4",        "mean"),
    綜合平均 = ("avg_score", "mean")
).round(2).reset_index()

summary.to_csv("survey_summary.csv", index=False, encoding="utf-8-sig")
print("✅ 輸出完成：survey_clean.csv、survey_summary.csv")
summary

---
<a id='homework'></a>
## 作業

完成以下二題，下次上課前上傳 `.ipynb` 檔。

### 作業 1：銷售資料清理與統計
對以下銷售資料進行清理（缺漏值、重複、型別），並計算：
1. 各地區的總銷售額與平均銷售額
2. 銷售額最高的前 3 名業務員
3. 輸出清理後的結果為 CSV

In [ ]:
sales_raw = pd.DataFrame({
    "salesperson": ["Amy", "Ben", "Amy", "Carol", "Dave", "Ben", "Eve", None],
    "region":      [" 北區", "南區 ", " 北區", "中區", "南區 ", "南區 ", "北區", "中區"],
    "amount":      [12000, 8500, 12000, 15000, None, 8500, 20000, 9000],
    "month":       [1, 1, 1, 2, 2, 1, 3, 3]
})

# 請在此作答


### 作業 2：成績單合併報表
將以下三個 DataFrame 合併成一張完整成績單，並新增：
- `total`：三科總分
- `rank`：班級排名（依 total 降序）
- `pass_all`：三科都 >= 60 才是 True

In [ ]:
info = pd.DataFrame({
    "id":    [1, 2, 3, 4, 5],
    "name":  ["Amy", "Ben", "Carol", "Dave", "Eve"],
    "class": ["A", "A", "B", "B", "A"]
})
math_df = pd.DataFrame({"id": [1, 2, 3, 4, 5], "math": [88, 55, 92, 70, 65]})
eng_df  = pd.DataFrame({"id": [1, 2, 3, 4, 5], "english": [76, 80, 90, 58, 72]})
sci_df  = pd.DataFrame({"id": [1, 2, 3, 4, 5], "science": [82, 78, 88, 65, 50]})

# 請在此作答
